# Checkpoint III · Examen blanc — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_3/03_examen_solutions.ipynb)

Les solutions des deux questions de code (CP3.6 et CP3.7), exécutées, puis l'enregistrement des réponses que vérifie la partie B du notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`). Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Partie A · Les deux questions de code

In [2]:
# Data and tools of the two code questions (run this cell first)
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

# CP3.6: 4 000 districts of California (1990 census)
california = wb.datasets.load_california()
FEATURES = california.columns[:8].tolist()                    # MedInc, HouseAge, AveRooms, AveBedrms, Population, ...
rows = np.sort(np.random.default_rng(2606).choice(len(california), 4000, replace=False))
X = california[FEATURES].to_numpy(dtype=float)[rows]          # shape (4000, 8)
y = california["MedHouseVal"].to_numpy(dtype=float)[rows]     # median house value (hundreds of thousands of $)
K_GRID = [1, 3, 5, 10, 20, 40]                                # numbers of neighbours tried


# CP3.7: the probabilities of a binary classifier, for 600 validation and 600 test examples
def make_scores(n, seed):
    """n labels (about 25 % of positives) and their probability of being positive (calibrated, 3 decimals)."""
    rng = np.random.default_rng(seed)
    labels = (rng.random(n) < 0.25).astype(int)
    score = rng.normal(np.where(labels == 1, 1.0, -1.0), 1.0)
    return np.round(1 / (1 + 3 * np.exp(-2 * score)), 3), labels


PROBA_VAL, Y_VAL = make_scores(600, 3707)
PROBA_TEST, Y_TEST = make_scores(600, 3708)
THRESHOLDS = np.round(np.arange(1, 100) / 100, 2)             # 0.01, 0.02, ..., 0.99
COST_FN, COST_FP = 5, 1                                       # a false negative costs 5, a false positive 1
_ORIGINALS = {"X": X.copy(), "y": y.copy(), "PROBA_VAL": PROBA_VAL.copy(), "Y_VAL": Y_VAL.copy(),
              "PROBA_TEST": PROBA_TEST.copy(), "Y_TEST": Y_TEST.copy()}   # part B restores them if a try changed them
print("X:", X.shape, "· y:", y.shape, "· validation:", PROBA_VAL.shape, "· positives:", int(Y_VAL.sum()))

X: (4000, 8) · y: (4000,) · validation: (600,) · positives: 150


### CP3.6 — Trouver trois fuites dans un script d'entraînement 🐛 ★★ ⏱️ 12 min · 2 points
**Objectif :** repérer ce qu'une évaluation a appris de son jeu de test, puis refaire l'évaluation sans fuite.

Un collègue prédit la valeur médiane des logements des 4 000 districts de `X` et `y` avec un kNN de régression, dont il choisit le nombre de voisins dans `K_GRID`. Sa fonction `colleague_report(X, y)`, dont les lignes sont marquées (A) à (F), annonce au client un $R^2$ de test de 0,93. Exécute-la (cellule suivante).

a) Trois de ses lignes contiennent une fuite de données : leurs lettres (sur ta feuille). **[0,6]**
b) Laquelle de ces fuites fausse vraiment son score, et pourquoi ? Pourquoi les deux autres ne le changent-elles presque pas ici, et pourquoi faut-il quand même les corriger ? (sur ta feuille) **[0,4]**
c) Écris `honest_report(X, y)`, qui fait le même travail **sans fuite** et renvoie `(best_k, test_r2)` : garde la transformation (B), qui n'est pas une fuite, et retire ce qui en est une ; découpe d'abord, avec le même `train_test_split(..., test_size=0.25, random_state=0)` ; choisis le nombre de voisins par une `GridSearchCV` à 5 folds (`cv=5`, score $R^2$ par défaut) sur l'entraînement seul, avec un `Pipeline` qui standardise puis applique le kNN ; mesure enfin le $R^2$ de test, une seule fois. La fonction ne modifie pas `X`. **[0,6]**
d) Les valeurs que renvoie ta fonction : $[\text{best\_k}, R^2_{\text{test}}]$ (3 décimales, sur ta feuille). **[0,2]**
e) Que dis-tu au client ? (sur ta feuille) **[0,2]**

In [3]:
# The colleague's code (do not change this cell: run it and read what it prints)
def colleague_report(X, y):
    X = np.column_stack([X, y / X[:, 2]])                                        # (A) a 9th feature: the value per room
    X[:, 4] = np.log1p(X[:, 4])                                                  # (B) the log of the population, row by row
    X = StandardScaler().fit_transform(X)                                        # (C) standardize the 9 columns
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)   # (D) split
    scores = {k: KNeighborsRegressor(n_neighbors=k).fit(X_train, y_train).score(X_test, y_test)
              for k in K_GRID}                                                   # (E) the R² of each k on the test
    best = max(scores, key=scores.get)
    return best, scores[best]                                                    # (F) the k kept and its test R²


best_colleague, r2_colleague = colleague_report(X, y)
print(f"the colleague keeps k = {best_colleague}, with a test R² of {r2_colleague:.3f}")

the colleague keeps k = 5, with a test R² of 0.934


In [4]:
# a) (A) builds a feature from the target y itself (y = value per room x AveRooms: the kNN reads the answer);
#    (C) learns the means and stds on all the districts, test included; (E) chooses k by its score ON THE TEST
#    and reports that same score. (B) is computed row by row: nothing learnt, no leak. (D) and (F) are fine.
def honest_report(X, y):
    """The same evaluation without leakage: returns (best_k, test_r2), see CP3.6 c)."""
    X = X.copy()                                                    # do not modify the caller's array
    X[:, 4] = np.log1p(X[:, 4])                                     # (B) kept: a row-by-row transformation
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
    search = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsRegressor()),
                          {"kneighborsregressor__n_neighbors": K_GRID}, cv=5).fit(X_train, y_train)
    return search.best_params_["kneighborsregressor__n_neighbors"], search.score(X_test, y_test)


best_6, r2_6 = honest_report(X, y)
print(f"honest report: k = {best_6}, test R² = {r2_6:.4f} (the colleague: k = {best_colleague}, {r2_colleague:.4f})")

honest report: k = 10, test R² = 0.7367 (the colleague: k = 5, 0.9337)


In [5]:
# Classic wrong versions of c), each recorded with its message
def variant_6(*, target_feature=False, keep_log=True, scaler_all=False, cv_score=False):
    """[best_k, R²] of honest_report written another (wrong) way."""
    Xv = X.copy()
    if target_feature:
        Xv = np.column_stack([Xv, y / Xv[:, 2]])
    if keep_log:
        Xv[:, 4] = np.log1p(Xv[:, 4])
    if scaler_all:
        Xv = StandardScaler().fit_transform(Xv)
    X_train, X_test, y_train, y_test = train_test_split(Xv, y, test_size=0.25, random_state=0)
    search = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsRegressor()),
                          {"kneighborsregressor__n_neighbors": K_GRID}, cv=5).fit(X_train, y_train)
    best = search.best_params_["kneighborsregressor__n_neighbors"]
    return [best, float(search.best_score_ if cv_score else search.score(X_test, y_test))]


wb.record("CP3.6d", [best_6, r2_6], decimals=4, mistakes={
    "ta fonction garde la feature (A), calculée à partir de y : c'est la fuite qui fait le score du collègue":
    variant_6(target_feature=True),
    "ta fonction a retiré la transformation (B) : l'énoncé demande de la garder, puisqu'elle se calcule ligne par ligne":
    variant_6(keep_log=False),
    "c'est le score moyen de validation croisée de la recherche (best_score_) : on demande le R² sur le jeu de test":
    variant_6(cv_score=True)})

WB_ANSWER {"decimals": 4, "element_coarse": [["c9ca32542aaab325ec23c98a4f028773db1edeba5223e9f4aa5308243cc0fb16"], ["3bc12bc1c1ef11d788ceab18c6de019b7b558516bc30aa754ce7334ca66c2a21"]], "element_hashes": ["92013ed8b6fd9c346610fff08afbbfdc5c98cbd93db01e521aab00e9b27ec33a", "aca42c303f4913a818e2e5e75e23763c63dc7ded5688578644ff03cd2ff8ea8b"], "hash": "833701cf17e52c6f9291826161108883b655ff3b2a6bee223d491016d0e39e5c", "id": "CP3.6d", "kind": "array", "mistakes": {"64a9b5e96048897c204400b3870a2f4c4edc8408156abe477ec0843f043fa18d": "ta fonction a retiré la transformation (B) : l'énoncé demande de la garder, puisqu'elle se calcule ligne par ligne", "71974a23d93d95034c112bc54ba9937ecf7f5d4fcde0cbbb2cbb2250c1aa05fe": "c'est le score moyen de validation croisée de la recherche (best_score_) : on demande le R² sur le jeu de test", "de5207e077da719148e5624736f088e365a7892aa373e4a7bc9a2e786234a9be": "ta fonction garde la feature (A), calculée à partir de y : c'est la fuite qui fait le score du coll

### CP3.7 — Coder le seuil de décision de coût minimal et le comparer au seuil théorique 🔨 ★★ ⏱️ 14 min · 2 points
**Objectif :** choisir un seuil de décision d'après le coût des erreurs, sur la validation, et le confronter au calcul théorique.

Avec NumPy seulement. `PROBA_VAL` donne, pour 600 exemples de validation, la probabilité d'être positif qu'annonce un classifieur, et `Y_VAL` leurs labels (0 ou 1) ; `PROBA_TEST` et `Y_TEST`, la même chose pour 600 exemples de test. Un faux négatif coûte `COST_FN = 5`, un faux positif `COST_FP = 1`. La règle de décision : classe 1 quand la probabilité est **supérieure ou égale** au seuil $t$.

a) Écris `total_cost(y_true, y_pred, cost_fn, cost_fp)`, qui renvoie le coût total (un `int`) de décisions 0/1. **[0,3]**
b) Écris `min_cost_threshold(proba, y_true, cost_fn, cost_fp, thresholds)`, qui renvoie le couple `(t, coût)` du seuil de plus petit coût parmi `thresholds` (le premier dans l'ordre de `thresholds` en cas d'égalité). Les deux fonctions ne modifient pas leurs arguments. **[0,5]**
c) Avec `THRESHOLDS` (0,01 ; 0,02 ; … ; 0,99), le seuil et le coût que donne ta fonction sur la validation : $[t, \text{coût}]$ (sur ta feuille). **[0,3]**
d) Le seuil théorique de coût moyen minimal, si les probabilités sont calibrées (4 décimales, sur ta feuille). Montre-le en comparant le coût moyen des deux décisions pour un exemple de probabilité $p$. **[0,3]**
e) Les coûts sur le test, avec le seuil de c), puis avec le seuil 0,5 : $[\text{coût}(t), \text{coût}(0{,}5)]$ (sur ta feuille). **[0,3]**
f) Pourquoi c) et d) ne donnent-ils pas le même seuil ? Lequel déploierais-tu, et pourquoi ? (sur ta feuille) **[0,3]**

In [6]:
def total_cost(y_true, y_pred, cost_fn, cost_fp):
    """Total cost (an int) of the 0/1 decisions y_pred, see CP3.7 a)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    false_negatives = np.sum((y_true == 1) & (y_pred == 0))
    false_positives = np.sum((y_true == 0) & (y_pred == 1))
    return int(cost_fn * false_negatives + cost_fp * false_positives)


def min_cost_threshold(proba, y_true, cost_fn, cost_fp, thresholds):
    """(t, cost) of the threshold of smallest total cost among thresholds (the first one if tied), CP3.7 b)."""
    proba = np.asarray(proba)
    costs = [total_cost(y_true, (proba >= t).astype(int), cost_fn, cost_fp) for t in thresholds]
    best = int(np.argmin(costs))                                   # argmin keeps the first minimum
    return float(thresholds[best]), costs[best]


t_7, cost_7 = min_cost_threshold(PROBA_VAL, Y_VAL, COST_FN, COST_FP, THRESHOLDS)
theory_7 = COST_FP / (COST_FP + COST_FN)                           # decide 1 when 5 p >= 1 - p
test_7 = [total_cost(Y_TEST, (PROBA_TEST >= t).astype(int), COST_FN, COST_FP) for t in (t_7, 0.5)]
print(f"validation: t = {t_7}, cost {cost_7} · theory {theory_7:.4f} · test: cost {test_7[0]} at t, {test_7[1]} at 0.5")

validation: t = 0.23, cost 173 · theory 0.1667 · test: cost 163 at t, 231 at 0.5


In [7]:
def best_with(rule, cost_fn=COST_FN, cost_fp=COST_FP):
    """[t, cost] on the validation with another (wrong) decision rule or other costs."""
    costs = [total_cost(Y_VAL, rule(PROBA_VAL, t).astype(int), cost_fn, cost_fp) for t in THRESHOLDS]
    best = int(np.argmin(costs))
    return [float(THRESHOLDS[best]), costs[best]]


wb.record("CP3.7c", [t_7, cost_7], decimals=4, mistakes={
    "la règle de l'énoncé décide « classe 1 » quand la probabilité est SUPÉRIEURE OU ÉGALE au seuil":
    best_with(lambda p, t: p > t),
    "les deux coûts sont échangés : un faux NÉGATIF coûte 5, un faux positif 1": best_with(lambda p, t: p >= t, 1, 5)})
wb.record("CP3.7e", test_7, decimals=4, mistakes={
    "l'ordre demandé est [coût au seuil de c), coût au seuil 0,5]": test_7[::-1],
    "ces coûts sont mesurés sur la validation : e) les demande sur le test":
    [total_cost(Y_VAL, (PROBA_VAL >= t).astype(int), COST_FN, COST_FP) for t in (t_7, 0.5)]})

WB_ANSWER {"decimals": 4, "element_coarse": [["af849392361c20fbe2c95e2b1861bcca0b950069521286e562c47959a0fc9b9d"], ["ccf98571a9fcaaf16f2ae9267846179260ef81486d8ed5e65eb303c590a7ef2a"]], "element_hashes": ["0b833c1bdbbd50fb6bdf21eb38071d5073c616a3866283d641cfc1327c390737", "9a373ed91fbf7753d6f70a76bdff3f50fcdc3f8aebc01a3134b82fbc892a7a35"], "hash": "572633b914006456ec73ec175799c5573e59b50321974c84590c8eed32227258", "id": "CP3.7c", "kind": "array", "mistakes": {"90f6ec2c538e0725ca12e8acf390a0f447ffe6bb759b50470029849c1c6ba9f2": "la règle de l'énoncé décide « classe 1 » quand la probabilité est SUPÉRIEURE OU ÉGALE au seuil", "cf0961b1af81b756bfc79c1b2f5819af4f9c1d74839f0829d0ce31f12e11d56c": "les deux coûts sont échangés : un faux NÉGATIF coûte 5, un faux positif 1"}, "shape": [2]}
📝 Ex CP3.7c : réponse enregistrée (array, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "element_coarse": [["0703bf6275da7e7fd84c397881be57048af1ea97e66e204088ac7e7afd8a5300"], ["d236e7084ac13bf1eadf507d8589157fd45

## Partie B · Les réponses vérifiées après l'examen

Chaque réponse est calculée à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher.

In [8]:
# The paper questions only use the numbers of their statements (01_examen_sujet.md)
import math

import numpy as np

# CP3.2
BILLS_2 = np.array([38, 46, 42, 50], dtype=float)
MU_2, SIGMA_2 = BILLS_2.mean(), BILLS_2.std()                      # ddof = 0, like StandardScaler
SIGMA1_2 = BILLS_2.std(ddof=1)
LOW_2, HIGH_2 = BILLS_2.min(), BILLS_2.max()

# CP3.3
X_3 = np.array([1, 2, 3, 5, 6, 8], dtype=float)
CLASSES_3 = np.array(list("AABBAB"))


def gini(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return float(1 - (p ** 2).sum())


def entropy_bits(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())


THRESHOLDS_3 = (X_3[:-1] + X_3[1:]) / 2


def weighted_gini(s, weighted=True):
    left, right = CLASSES_3[X_3 <= s], CLASSES_3[X_3 > s]
    if not weighted:
        return (gini(left) + gini(right)) / 2
    return (left.size * gini(left) + right.size * gini(right)) / X_3.size


BEST_3 = float(THRESHOLDS_3[np.argmin([weighted_gini(s) for s in THRESHOLDS_3])])

# CP3.4
TRAIN_4 = np.array([[4000, 42], [3900, 48], [4300, 39], [4400, 46], [4450, 50]], dtype=float)
SPECIES_4 = np.array(["Adélie", "Chinstrap", "Adélie", "Chinstrap", "Chinstrap"])
Q_4 = np.array([3800, 46], dtype=float)
MEAN_4, STD_4 = np.array([4210, 45], dtype=float), np.array([220, 4], dtype=float)


def neighbours_4(standardize):
    A, q = (TRAIN_4 - MEAN_4) / STD_4 if standardize else TRAIN_4, (Q_4 - MEAN_4) / STD_4 if standardize else Q_4
    order = np.argsort(np.sqrt(((A - q) ** 2).sum(axis=1)))[:3]
    return (order + 1).tolist(), max(set(SPECIES_4[order]), key=list(SPECIES_4[order]).count)


# CP3.5
EPS_5 = 3 / 10
ALPHA_5 = math.log((1 - EPS_5) / EPS_5)
WRONG_5, RIGHT_5 = 0.1 * math.exp(ALPHA_5), 0.1
TOTAL_5 = 3 * WRONG_5 + 7 * RIGHT_5
X_5, Y_5 = np.array([1, 2, 3, 4], dtype=float), np.array([2, 4, 9, 11], dtype=float)
F0_5 = float(Y_5.mean())
R_5 = Y_5 - F0_5


def stump_5(r):
    """Regression stump on x_5: [threshold, left value, right value] of smallest sum of squares."""
    best = None
    for s in (X_5[:-1] + X_5[1:]) / 2:
        left, right = r[X_5 <= s], r[X_5 > s]
        sse = ((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum()
        if best is None or sse < best[0]:
            best = (sse, [float(s), float(left.mean()), float(right.mean())])
    return best[1]


STUMP_5 = stump_5(R_5)


def f1_5(eta):
    return F0_5 + eta * np.where(X_5 <= STUMP_5[0], STUMP_5[1], STUMP_5[2])


# CP3.11
TP_11, FP_11 = 45, 60 - 45
FN_11, TN_11 = 75 - TP_11, 500 - 75 - FP_11
P_11, R_11 = TP_11 / (TP_11 + FP_11), TP_11 / (TP_11 + FN_11)

# CP3.13


def var_mean(sigma2, rho, m):
    return rho * sigma2 + (1 - rho) * sigma2 / m


# CP3.14
CANDIDATES_14 = 4 * 3 * 5
FITS_14 = CANDIDATES_14 * 5 + 1

**CP3.1 — Prédire la sortie de six extraits scikit-learn (une lettre par extrait)**

In [9]:
wb.record("CP3.1a", "A", mistakes={"un classifieur binaire n'a qu'UN score par exemple : decision_function renvoie un tableau à une dimension": "B",
                     "relis la forme : à deux classes, decision_function renvoie un tableau à UNE dimension, pas une colonne": "C",
                     "la forme d'une sortie a d'abord le nombre d'exemples ; relis ce que rend decision_function à deux classes": "D"})
wb.record("CP3.1b", "C", mistakes={"predict_proba donne une distribution de probabilité par exemple : que vaut la somme de chaque ligne ?": "A",
                     "chaque ligne de predict_proba contient les probabilités des trois classes : quelle est leur somme ?": "B",
                     "une forêt fait la moyenne des probabilités de ses arbres : elle a bien un predict_proba": "D"})
wb.record("CP3.1c", "B", mistakes={"une ligne de zéros, c'est le réglage handle_unknown « ignore » : quel est le réglage PAR DÉFAUT de OneHotEncoder ?": "A",
                     "un encodeur appris ne crée jamais de colonne après fit : relis le réglage par défaut de handle_unknown": "C",
                     "relis le réglage par défaut de handle_unknown de OneHotEncoder": "D"})
wb.record("CP3.1d", "D", mistakes={"la moyenne et l'écart-type viennent de X_train : sur des données nouvelles, la moyenne n'est pas exactement 0": "A",
                     "relis ce que transform fait des moyennes : chaque colonne est centrée sur la moyenne d'ENTRAÎNEMENT": "B",
                     "transform s'applique ligne par ligne : il n'a pas besoin du même nombre de lignes qu'à l'entraînement": "C"})
wb.record("CP3.1e", "A", mistakes={"le pipeline a deux étapes : la grille doit dire à quelle étape appartient le paramètre": "B",
                     "compte les tirets bas entre le nom de l'étape et le paramètre": "C",
                     "le nom d'une étape est celui que tu lui as donné dans le Pipeline": "D"})
wb.record("CP3.1f", "D", mistakes={"quand toutes les entrées sont des listes, la recherche au hasard tire-t-elle avec ou sans remise ?": "A",
                     "scikit-learn ne s'arrête pas : relis ce qu'il fait quand n_iter dépasse la taille de la grille": "B",
                     "une liste ne se tire pas comme une loi continue : RandomizedSearchCV choisit parmi les valeurs données": "C"})

WB_ANSWER {"hash": "b3cafd16dec7e905705ef7bf48034b479efecb7f69e7a67ecb4878b224ed9cc8", "id": "CP3.1a", "kind": "str", "mistakes": {"40e9942a40f57849bbf09e173afc79dd69dbb430e6d66e1598f6179f3011ba51": "un classifieur binaire n'a qu'UN score par exemple : decision_function renvoie un tableau à une dimension", "85fdb6f3184244f5720704246473135699b3661d75bd36b78b5056215fc26a28": "la forme d'une sortie a d'abord le nombre d'exemples ; relis ce que rend decision_function à deux classes", "e692d9a63b543574a0c0dbf16cead692f00d790694c536fc81bc4d9c361828ff": "relis la forme : à deux classes, decision_function renvoie un tableau à UNE dimension, pas une colonne"}}
📝 Ex CP3.1a : réponse enregistrée (str).
WB_ANSWER {"hash": "d5f4caac0de974fcd0fb8400de75352d9f4d523d1061012602af6c56f76f223a", "id": "CP3.1b", "kind": "str", "mistakes": {"05e2159193cea54e13de2ea119b43653055610adf3641ac40599cfa43128009b": "une forêt fait la moyenne des probabilités de ses arbres : elle a bien un predict_proba", "34325300

**CP3.2 — Standardiser et encoder un mini-tableau**

In [10]:
wb.record("CP3.2a", [MU_2, SIGMA_2], decimals=4, mistakes={"StandardScaler divise par n, pas par n − 1 : relis l'énoncé": [MU_2, SIGMA1_2],
                     "c'est la variance : l'écart-type en est la racine": [MU_2, SIGMA_2 ** 2]})
wb.record("CP3.2b", (46 - MU_2) / SIGMA_2, decimals=3, mistakes={"StandardScaler divise par l'écart-type calculé avec n, pas n − 1": (46 - MU_2) / SIGMA1_2})
wb.record("CP3.2c", (54 - MU_2) / SIGMA_2, decimals=3, mistakes={"StandardScaler divise par l'écart-type calculé avec n, pas n − 1": (54 - MU_2) / SIGMA1_2,
                     "le scaler a appris sur les quatre manchots d'entraînement seulement : T1 n'entre pas dans la moyenne": (54 - np.append(BILLS_2, 54).mean()) / np.append(BILLS_2, 54).std()})
wb.record("CP3.2d", (54 - LOW_2) / (HIGH_2 - LOW_2), decimals=3, mistakes={"MinMaxScaler ne borne pas les données nouvelles par défaut (clip=False)": 1.0,
                     "le min-max divise par l'étendue (max − min) de l'entraînement": (54 - LOW_2) / HIGH_2})
wb.record("CP3.2e", [0, 1, 0], mistakes={"les catégories sont rangées dans l'ordre alphabétique, pas dans l'ordre d'apparition": [1, 0, 0]})
wb.record("CP3.2f", [0, 0, 0])
wb.record("CP3.2g", [(41 - MU_2) / SIGMA_2, 0, 0, 0], decimals=3, mistakes={"StandardScaler divise par l'écart-type calculé avec n, pas n − 1": [(41 - MU_2) / SIGMA1_2, 0, 0, 0],
                     "relis l'ordre des colonnes : d'abord le z-score du bec, puis le one-hot": [0, 0, 0, (41 - MU_2) / SIGMA_2]})

WB_ANSWER {"decimals": 4, "element_coarse": [["0c44fbc4446c58466ed896ba7bb1bc64b8aaa4b3d8bb2de0bf3c18b36bc7eadf"], ["e30956363f7bb5bb85dac7542e57db67363d806e8db48c3a985e4a948fea5090"]], "element_hashes": ["297c925699033baf99a84c65c28bdbaebe12981405368d4f5bb68e2cdd1a12ff", "1399fa6a2b82d616edde59533443d0d5bab445c54f07eea12c1d770fba57e76b"], "hash": "45a334d46b6d1c214527db5d69fc4578fc67e9e7006f1f05aeeb2a1a24e555ba", "id": "CP3.2a", "kind": "array", "mistakes": {"cbb70cc6f4092edbd9bb9167c7f0f09cde7512602ac66fd8e69e7ffe642f7004": "c'est la variance : l'écart-type en est la racine", "f7ad0970f1b654471d7a498c8ca8b99fea02c94cc6ff31db8c654b94ced813a3": "StandardScaler divise par n, pas par n − 1 : relis l'énoncé"}, "shape": [2]}
📝 Ex CP3.2a : réponse enregistrée (array, 4 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "d86fce1f528157d5ff0ad6ceae032a13369ae50735eb00bf7055fd022ba85a45", "hash_coarse": "f91231d516956cbe4e183838f852434f5a39be1fc7839e6fde4de5b47300daa1", "id": "CP3.2b", "kind": "f

**CP3.3 — Gini et meilleur seuil**

In [11]:
wb.record("CP3.3a", gini(CLASSES_3), decimals=3, mistakes={"c'est l'entropie de la racine, en bits : on demande son Gini, 1 − Σ p²": 1.0,
                     "le Gini est 1 − Σ p², la somme portant sur les deux classes": 0.25})
wb.record("CP3.3b", THRESHOLDS_3.size, fractional="on demande un nombre de seuils", mistakes={"un seuil se place ENTRE deux abscisses consécutives": 6})
wb.record("CP3.3c", weighted_gini(4.0), decimals=3)
wb.record("CP3.3d", BEST_3, decimals=1, choices=THRESHOLDS_3.tolist(), mistakes={f"s = {s:g} : calcule son Gini pondéré et compare-le à celui des autres seuils".replace(".", ","): float(s)
                     for s in THRESHOLDS_3 if s != BEST_3})
wb.record("CP3.3e", [weighted_gini(BEST_3), gini(CLASSES_3) - weighted_gini(BEST_3)], decimals=3, mistakes={"le Gini pondéré pèse chaque enfant par son NOMBRE d'exemples, pas une moyenne simple": [weighted_gini(BEST_3, weighted=False), gini(CLASSES_3) - weighted_gini(BEST_3, weighted=False)]})
wb.record("CP3.3f", entropy_bits(CLASSES_3[X_3 > BEST_3]), decimals=3, mistakes={"l'entropie demandée est en bits : log₂, pas ln": entropy_bits(CLASSES_3[X_3 > BEST_3]) * math.log(2),
                     "c'est le Gini de cet enfant : on demande son entropie, −Σ p log₂ p": gini(CLASSES_3[X_3 > BEST_3])})

WB_ANSWER {"decimals": 3, "hash": "287a70e1765059aaf4635289a24f29f754098fa58b32f1f273cec75593eb3dc5", "hash_coarse": "2b2c168887d094a7885eb8d1a64cca39a523b16db6cdf05b79a162d0514ac8f1", "id": "CP3.3a", "kind": "float", "magnitude": -1, "mistakes": {"b4a2cf12423deac51f3fad0fa05fbf68e4f8bad4cb6ed92ea7a9974a75f04568": "c'est l'entropie de la racine, en bits : on demande son Gini, 1 − Σ p²", "e2d1731c1bed0a202da4fb8f90b0b7b55ef85b63739b1f5e1eba340c5534561f": "le Gini est 1 − Σ p², la somme portant sur les deux classes"}, "sign": 1}
📝 Ex CP3.3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"fractional": "on demande un nombre de seuils", "hash": "f87e6197ed539c98148d8f8f00a62e94fcf886fd6dd7c0969c4903f20bf35951", "id": "CP3.3b", "kind": "int", "magnitude": 0, "mistakes": {"0c8fa34fe7d7bcd3c4984591fad42335c32b0e68e71b65d49c96541c2f9abbb4": "un seuil se place ENTRE deux abscisses consécutives"}, "sign": 1}
📝 Ex CP3.3b : réponse enregistrée (int).
WB_ANSWER {"decimals": 3, "hash": "4e9

**CP3.4 — kNN avant et après standardisation**

In [12]:
wb.record("CP3.4a", neighbours_4(False)[0], mistakes={"la liste va du plus proche au moins proche": neighbours_4(False)[0][::-1]})
wb.record("CP3.4b", neighbours_4(False)[1], mistakes={"relis le vote : quelle espèce a la majorité parmi les trois voisins de a) ?": "Chinstrap"})
wb.record("CP3.4c", ((Q_4 - MEAN_4) / STD_4).tolist(), decimals=3, mistakes={"un z-score retire la moyenne PUIS divise par l'écart-type : (x − μ)/σ": ((MEAN_4 - Q_4) / STD_4).tolist()})
wb.record("CP3.4d", neighbours_4(True)[0], mistakes={"la liste va du plus proche au moins proche": neighbours_4(True)[0][::-1],
                     "ce sont encore les voisins des mesures brutes : standardise aussi les cinq manchots d'entraînement, avec les mêmes statistiques": neighbours_4(False)[0]})
wb.record("CP3.4e", neighbours_4(True)[1], mistakes={"relis le vote : quelle espèce a la majorité parmi les trois voisins de d) ?": "Adélie"})

WB_ANSWER {"decimals": 0, "element_hashes": ["55f3eef983312fa1a737d0846a074ba1e00de0569dd977ac9958a1208d836b08", "05b285a983d1cb582187660983885e111f3b47383883e66d0ab87770e15c66d0", "38fb7b437217236a36f640cd3c6c4c9b3b5e4efd8d3e469c3a2a9e91d7d32a03"], "hash": "d4d0527264c4592f6a85a01f7da041773e3ecd805c0aec832b6160f70f02e197", "id": "CP3.4a", "integer": true, "kind": "array", "mistakes": {"bf975b66e55d1141f53dedb5ebd104c8468844898a4dd23b5f463c599cf2962a": "la liste va du plus proche au moins proche"}, "shape": [3]}
📝 Ex CP3.4a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"hash": "aea38d27ca4c3206aef7b60c045b0d6335289d7d1b374fc918a51c5082550026", "id": "CP3.4b", "kind": "str", "mistakes": {"5f329fbe1fe444cedca0339054bda0fc66a40ab00643269fcb997a6dfd89a23c": "relis le vote : quelle espèce a la majorité parmi les trois voisins de a) ?"}}
📝 Ex CP3.4b : réponse enregistrée (str).
WB_ANSWER {"decimals": 3, "element_coarse": [["8df537a125b94c3173f9dc4bc97cf5eb370a0cb261aa78b8cbc4bcf01

**CP3.5 — AdaBoost et gradient boosting**

In [13]:
wb.record("CP3.5a", EPS_5, decimals=2)
wb.record("CP3.5b", ALPHA_5, decimals=3, mistakes={"la fiche du ch. 14 (et scikit-learn) n'a pas le facteur ½ des textes à labels ±1 : α = ln((1 − ε)/ε)": ALPHA_5 / 2,
                     "ln est le logarithme népérien, pas log₁₀": math.log10((1 - EPS_5) / EPS_5),
                     "le rapport est (1 − ε)/ε, pas ε/(1 − ε)": -ALPHA_5})
wb.record("CP3.5c", WRONG_5 / TOTAL_5, decimals=4, mistakes={"n'oublie pas la normalisation : divise par la somme des nouveaux poids": WRONG_5,
                     "seuls les exemples MAL classés sont multipliés par e^α": 0.1 * math.exp(-ALPHA_5) / (3 * 0.1 * math.exp(-ALPHA_5) + 0.7)})
wb.record("CP3.5d", RIGHT_5 / TOTAL_5, decimals=4, mistakes={"n'oublie pas la normalisation : divise par la somme des nouveaux poids": RIGHT_5})
wb.record("CP3.5e", ALPHA_5 + math.log(4 - 1), decimals=3, mistakes={"le terme de SAMME est ln(K − 1), pas ln(K)": ALPHA_5 + math.log(4),
                     "avec K = 4, ln(K − 1) = ln 3": ALPHA_5 + math.log(2)})
wb.record("CP3.5f", F0_5, decimals=2)
wb.record("CP3.5g", R_5.tolist(), decimals=2, mistakes={"un résidu est y − F₀, pas F₀ − y": (-R_5).tolist()})
wb.record("CP3.5h", STUMP_5, decimals=2, mistakes={"les feuilles prédisent la moyenne des RÉSIDUS, pas celle des y": [STUMP_5[0], float(Y_5[X_5 <= STUMP_5[0]].mean()), float(Y_5[X_5 > STUMP_5[0]].mean())]})
wb.record("CP3.5i", float(f1_5(0.1)[0]), decimals=2, mistakes={"le learning rate multiplie la valeur de la feuille : F₁ = F₀ + η × feuille": float(f1_5(1.0)[0]),
                     "F₁ part de F₀, pas de 0 : F₁ = F₀ + η × feuille": float(0.1 * STUMP_5[1])})
wb.record("CP3.5j", float(np.mean((Y_5 - f1_5(0.1)) ** 2)), decimals=4, mistakes={"c'est la MSE de départ, avec F₀ : on la demande après l'étape": float(np.mean(R_5 ** 2)),
                     "c'est la MSE avec η = 1 : l'énoncé prend η = 0,1": float(np.mean((Y_5 - f1_5(1.0)) ** 2)),
                     "c'est une RMSE : on demande la MSE, sans racine": float(np.sqrt(np.mean((Y_5 - f1_5(0.1)) ** 2)))})

WB_ANSWER {"decimals": 2, "hash": "571f7489d2ff54f0e61526eed5cbde089e55819fb8d8a838e8b23b194b3bd76f", "hash_coarse": "38fdf9e19a7f405bf36611bbbc0bcc8332c6a4721a4ef1c69b2badcf09bfcabf", "id": "CP3.5a", "kind": "float", "magnitude": -1, "sign": 1}


📝 Ex CP3.5a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "1f9033ec440ede8ef1b8daed31ab66e31a61fbac886a797dac6b206410306209", "hash_coarse": "2f6790a17ba0f5a4cb3e9f12e955e4e793262d501fc6cdbcce08b4ffe053abf6", "id": "CP3.5b", "kind": "float", "magnitude": -1, "mistakes": {"308218da421e10e3ba6e003419697d9fe10a30baa1fe656b8f7a707909f11ce3": "la fiche du ch. 14 (et scikit-learn) n'a pas le facteur ½ des textes à labels ±1 : α = ln((1 − ε)/ε)", "96c396013562e229ccaa6c69b0651d644e1709bbc082de3bf454f5167888207f": "ln est le logarithme népérien, pas log₁₀", "b966d3f09451d93e6f267176abb0fea0e81895005c5b41f3d6855bd0b70cb6c7": "le rapport est (1 − ε)/ε, pas ε/(1 − ε)"}, "sign": 1}
📝 Ex CP3.5b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "1a495eb792296bbff1e006fc5853d088f2b2552130fddcb4c8b8662ec422afc2", "hash_coarse": "070a49bcb48591ee4f33bcac56a0b74c87729f5364fd82a87121facc1a2742a3", "id": "CP3.5c", "kind": "float", "magnitude

**CP3.6 — Les lettres des trois fuites (a)**

In [14]:
wb.record("CP3.6a", "ACE", mistakes={"il manque une fuite : la ligne (E) choisit k d'après son score sur le TEST": "AC",
                     "il manque une fuite : la ligne (C) apprend ses moyennes et ses écarts-types sur TOUS les districts, test compris": "AE",
                     "il manque une fuite : la ligne (A) construit une feature à partir de y, la cible": "CE",
                     "une ligne de trop : (B) se calcule ligne par ligne, sans rien apprendre des autres districts": "ABCE",
                     "une ligne de trop : (D) découpe ; c'est ce qu'elle reçoit qui pose problème, pas le découpage": "ACDE",
                     "une ligne de trop : (F) rend ce que (E) a calculé ; la fuite est dans le choix de k": "ACEF",
                     "parmi les lignes A, B et C, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "ABC",
                     "parmi les lignes A, B et D, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "ABD",
                     "parmi les lignes A, B et E, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "ABE",
                     "parmi les lignes A, B et F, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "ABF",
                     "parmi les lignes A, C et D, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "ACD",
                     "parmi les lignes A, C et F, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "ACF",
                     "parmi les lignes A, D et E, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "ADE",
                     "parmi les lignes A, D et F, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "ADF",
                     "parmi les lignes A, E et F, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "AEF",
                     "parmi les lignes B, C et D, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "BCD",
                     "parmi les lignes B, C et E, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "BCE",
                     "parmi les lignes B, C et F, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "BCF",
                     "parmi les lignes B, D et E, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "BDE",
                     "parmi les lignes B, D et F, aucune n'est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "BDF",
                     "parmi les lignes B, E et F, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "BEF",
                     "parmi les lignes C, D et E, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "CDE",
                     "parmi les lignes C, D et F, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "CDF",
                     "parmi les lignes C, E et F, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "CEF",
                     "parmi les lignes D, E et F, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test": "DEF"})

WB_ANSWER {"hash": "c01ba03f4b10eea6f73c6d305be3b9ef3e56b8b470cc1d6bf3db0774bd561535", "id": "CP3.6a", "kind": "str", "mistakes": {"0102f9bbd2a26c4048ae1fb9e84ac5da4e10f7de1f679370adf43c2fb55a4feb": "il manque une fuite : la ligne (A) construit une feature à partir de y, la cible", "03d2caf922f497cf488c9a76b762e0a994a97c32f93794049a00067cfaee2014": "parmi les lignes B, C et E, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test", "273a35ebf50983ad16d9ca55d7186a96c0c28b4c0883f9f461512b880aa5b3b9": "parmi les lignes A, E et F, deux seulement sont des fuites : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test", "40a89bcd78abf58ac6cfe6706ec349b1370a1824700c89c23285bda6b5ee3d42": "parmi les lignes A, B et D, une seule est une fuite : demande-toi pour chaque ligne si elle apprend quelque chose de la cible ou des districts de test", "4287d3ce3d4a5d9964979d730abc0c1bd4e7

**CP3.7 — Le seuil théorique (d)**

In [15]:
wb.record("CP3.7d", COST_FP / (COST_FP + COST_FN), decimals=4, mistakes={"les deux coûts sont échangés : décider « 1 » ne coûte que si l'exemple est négatif, décider « 0 » que s'il est positif": COST_FN / (COST_FP + COST_FN),
                     "le rapport des coûts n'est pas le seuil : écris le coût moyen de chaque décision pour une probabilité p, puis cherche où ils s'égalent": COST_FP / COST_FN})

WB_ANSWER {"decimals": 4, "hash": "2d578326d5c3cdaef573de2358a5a7678e2fb32c376a5cfa450301c6a52d6a1c", "hash_coarse": "9b7f513a75b057f75f3940e7644604667eceedf00dd7cd0c8a4e336f4841d7be", "id": "CP3.7d", "kind": "float", "magnitude": -1, "mistakes": {"52f75a05244a886e83c5e1e2c31628e98e219f09090c2eb41a9887d65b210a7a": "le rapport des coûts n'est pas le seuil : écris le coût moyen de chaque décision pour une probabilité p, puis cherche où ils s'égalent", "9112f83429848eaa99d96c18610af4f6738481f200823766a493da8c64a44c68": "les deux coûts sont échangés : décider « 1 » ne coûte que si l'exemple est négatif, décider « 0 » que s'il est positif"}, "sign": 1}
📝 Ex CP3.7d : réponse enregistrée (float, 4 décimale(s)).


**CP3.8 — Lire les courbes de la figure**

In [16]:
wb.record("CP3.8a", 2, choices=[1, 2], mistakes={"la courbe d'entraînement est celle qu'un arbre plus profond fait toujours baisser, jusqu'à apprendre par cœur": 1})
wb.record("CP3.8b", "B", mistakes={"cherche le point le plus bas de la courbe de VALIDATION, pas celui où les deux courbes sont proches": "A",
                     "relis la courbe de validation : où atteint-elle son minimum ?": "C",
                     "l'erreur d'entraînement baisse toujours : la profondeur se choisit sur la courbe de validation": "D"})
wb.record("CP3.8c", "A", mistakes={"à max_depth=2, les deux erreurs sont hautes et proches : est-ce de l'overfitting ?": "B",
                     "à max_depth=2, l'erreur de validation est encore loin de son minimum": "C"})
wb.record("CP3.8d", "B", mistakes={"à max_depth=18, l'erreur d'entraînement est presque nulle : est-ce de l'underfitting ?": "A",
                     "à max_depth=18, l'écart entre les deux courbes est grand et la validation remonte": "C"})
wb.record("CP3.8e", 1.0, decimals=1, choices=[1.0, 0.1], mistakes={"avec le learning rate 0,1, l'erreur de validation ne remonte pas (elle baisse de plus en plus lentement jusqu'à 400 arbres) : laquelle remonte ?": 0.1})
wb.record("CP3.8f", "B", mistakes={"regarde où la courbe de validation de ce learning rate atteint son point le plus bas": "A",
                     "après son minimum, la courbe de validation remonte : où est ce minimum ?": "C",
                     "la courbe de validation de ce learning rate remonte bien avant la fin": "D"})
wb.record("CP3.8g", True)

WB_ANSWER {"choices": ["1", "2"], "hash": "fcbcf23841eb6361a12d3e6a3291732727a02e6c718863a65d63ef8547f53770", "id": "CP3.8a", "kind": "int", "magnitude": 0, "mistakes": {"fbc175bc9e6b5a03f3723864dd02b2ab68c62e99069c8ae3b076f15c15551f73": "la courbe d'entraînement est celle qu'un arbre plus profond fait toujours baisser, jusqu'à apprendre par cœur"}, "sign": 1}
📝 Ex CP3.8a : réponse enregistrée (int).
WB_ANSWER {"hash": "8a3fe020b4ccfb452bc8abf82d534f1719696c55634b0a8110b623c197aec76d", "id": "CP3.8b", "kind": "str", "mistakes": {"0ec99253e67ea92f608ab0b5c1d018e8d8e5c737d266078a953bfa4fabb68904": "cherche le point le plus bas de la courbe de VALIDATION, pas celui où les deux courbes sont proches", "e10dad0cfa7ae926c4f0855f2e4097f4b25e7783684f39a45cd6d9b15b726159": "relis la courbe de validation : où atteint-elle son minimum ?", "f3484d43e3157fc059065b7ba6d50a029a585265d8d5a25c5f2937f2d0924a36": "l'erreur d'entraînement baisse toujours : la profondeur se choisit sur la courbe de validati

**CP3.11 — Parties I et II**

In [17]:
wb.record("CP3.11a", [TP_11, FP_11, FN_11, TN_11], mistakes={"FP : les transactions signalées qui ne sont pas des fraudes ; FN : les fraudes non signalées": [TP_11, FN_11, FP_11, TN_11],
                     "les 500 transactions se répartissent dans les quatre cases : TN = 500 − TP − FP − FN": [TP_11, FP_11, FN_11, 500 - 75]})
wb.record("CP3.11b", P_11, decimals=3, mistakes={"precision = TP / (TP + FP) : parmi les transactions SIGNALÉES": R_11})
wb.record("CP3.11c", R_11, decimals=3, mistakes={"recall = TP / (TP + FN) : parmi les VRAIES fraudes": P_11})
wb.record("CP3.11d", 2 * P_11 * R_11 / (P_11 + R_11), decimals=3, mistakes={"le F1 est la moyenne HARMONIQUE de la precision et du recall": (P_11 + R_11) / 2})
wb.record("CP3.11e", (TP_11 + TN_11) / 500, decimals=3, mistakes={"l'accuracy compte les bonnes réponses, positives ET négatives, parmi les 500": TP_11 / 500})
wb.record("CP3.11f", [48, 12], mistakes={"un fold de validation contient 1/5 des exemples, et la stratification y met 1/5 de chaque classe": [48, 60],
                     "relis l'ordre demandé : [taille, classe 1]": [12, 48]})
wb.record("CP3.11g", [192, 48], mistakes={"l'entraînement d'un tour, ce sont les QUATRE autres folds": [48, 12],
                     "relis l'ordre demandé : [taille, classe 1]": [48, 192]})
wb.record("CP3.11h", 5 * 3, mistakes={"chaque répétition refait les 5 folds : multiplie": 5 + 3,
                     "cross_val_score ne réentraîne pas de modèle final : il ne compte que les tours": 16})

WB_ANSWER {"decimals": 0, "element_hashes": ["fce6ddc14e856558f8fff46e8d0d8c56c22df1b4bf045bab1e70e8ea1fb7e487", "d07bf67577e5943cb2fedd230c1215262aad739fef5b44d2039391138383a0ea", "fa1116f1964d9858c58f0d671691d3af023dc6e2e937ee82f5ac4ec814a0a1cf", "6f6dd41fb9241564c7c1a59740960309f75299053bb40ffb2a1876526a4a5a89"], "hash": "a3f0f9fe45b09e3b50ba621b95d0ab7f8b7103714003c38acd5367276e79978f", "id": "CP3.11a", "integer": true, "kind": "array", "mistakes": {"01dd20383649fba75a8ad537fed2c6a649e07413503cc993a9182bdcbf84b044": "les 500 transactions se répartissent dans les quatre cases : TN = 500 − TP − FP − FN", "74f9f78bfe4aea1322aa78c25be02af7e21b84ba7ad432add3452183f8fddc45": "FP : les transactions signalées qui ne sont pas des fraudes ; FN : les fraudes non signalées"}, "shape": [4]}
📝 Ex CP3.11a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "8b72aa3224079f6d0cf1afef85ad07b553cc10b875e1cc20bed5653a16c2212b", "hash_coarse": "0e563f49f6ce6221414aeaf6513578

**CP3.12 — Vrai ou faux justifiés (le verdict seulement ; les justifications se notent avec le corrigé)**

In [18]:
wb.record("CP3.12a", False)
wb.record("CP3.12b", True)
wb.record("CP3.12c", False)
wb.record("CP3.12d", True)
wb.record("CP3.12e", False)
wb.record("CP3.12f", True)

WB_ANSWER {"hash": "a50712cd29ec5113fdfd2f5f15d9cb15385d870455c39a54713755d22f447ace", "id": "CP3.12a", "kind": "bool"}
📝 Ex CP3.12a : réponse enregistrée (bool).
WB_ANSWER {"hash": "6d8c28d395010c62ce11fc0f78d1f82cda17c52ea7b575b6a1751799c35a86d0", "id": "CP3.12b", "kind": "bool"}
📝 Ex CP3.12b : réponse enregistrée (bool).
WB_ANSWER {"hash": "c4e5b57862407fc51f7710e8968a63a7fdbfd5a92cc503bf4b4bdad90d6f79f1", "id": "CP3.12c", "kind": "bool"}
📝 Ex CP3.12c : réponse enregistrée (bool).
WB_ANSWER {"hash": "360cc8cf37c74c4edd9a47bca47bf9fc76ab68098d945c8016d0dbf24ea0f052", "id": "CP3.12d", "kind": "bool"}
📝 Ex CP3.12d : réponse enregistrée (bool).
WB_ANSWER {"hash": "7b4c7d6c36a34240b82470976149728a8ffd10be8a9f2903924637a347b60d11", "id": "CP3.12e", "kind": "bool"}
📝 Ex CP3.12e : réponse enregistrée (bool).
WB_ANSWER {"hash": "6d2228dda90b9e1236820fd5f2cc87aa8509a0dbde0ed5644a1150fc882ab993", "id": "CP3.12f", "kind": "bool"}
📝 Ex CP3.12f : réponse enregistrée (bool).


**CP3.13 — Variance d'une moyenne de modèles corrélés**

In [19]:
wb.record("CP3.13b", var_mean(4, 0.25, 10), decimals=3, mistakes={"c'est la variance de modèles indépendants, σ²/M : la corrélation ρ ajoute ρ σ²": 4 / 10,
                     "relis la formule : le second terme est (1 − ρ) σ² / M": 0.25 * 4 + 4 / 10})
wb.record("CP3.13c", 0.25 * 4, decimals=3, mistakes={"c'est le terme qui disparaît : quand M grandit, (1 − ρ) σ²/M tend vers 0, et c'est ρ σ² qui reste": 0.75 * 4,
                     "la variance ne tend pas vers 0 : seul le second terme disparaît, le premier, ρ σ², reste": 0.0})
wb.record("CP3.13d", min(m for m in range(1, 1000) if var_mean(4, 0.25, m) <= 1.08), fractional="on demande un nombre d'arbres", mistakes={"calcule la variance pour ce M : elle dépasse encore 1,08 (M doit être un entier, arrondi vers le haut)": 37,
                     "le second terme est (1 − ρ) σ² / M, pas σ² / M": min(m for m in range(1, 1000) if 1 + 4 / m <= 1.08)})
wb.record("CP3.13e", [0.1 * 5, var_mean(5, 0.1, 30)], decimals=3, mistakes={"relis l'ordre demandé : [limite, variance pour M = 30]": [var_mean(5, 0.1, 30), 0.1 * 5]})

WB_ANSWER {"decimals": 3, "hash": "b4a6892092554cb30623eb0aa8d8d88bf2f68bd54384a8722e94c3cc70cce5a6", "hash_coarse": "266f90f199db9372693808412e0867223576fa1abfba50876de9b7a5b2501cb4", "id": "CP3.13b", "kind": "float", "magnitude": 0, "mistakes": {"59b0ca711b86f52f25645c9e5c3b1a72f399b5402c76fc28af1862f621459d47": "c'est la variance de modèles indépendants, σ²/M : la corrélation ρ ajoute ρ σ²", "5a9b255b9fd1ce67b1ee5097f7896d306a9b70c8eb87d223672d36d573078fdb": "relis la formule : le second terme est (1 − ρ) σ² / M"}, "sign": 1}
📝 Ex CP3.13b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "814559d97de01443c3f90d39313274e835ed7c335ab420262f07c4f5b8fc8da1", "hash_coarse": "641c1a1bb86bc6c3f6dc823c7696e37e29891d1d869d0c2f88600cb8eae4503d", "id": "CP3.13c", "kind": "float", "magnitude": 0, "mistakes": {"12acb9a7bd27d7724c23441e30d7c0e1f79d3154c5aa1077194d15cc3367fc2e": "c'est le terme qui disparaît : quand M grandit, (1 − ρ) σ²/M tend vers 0, et c'est ρ σ² q

**CP3.14 — Coût d'une recherche en grille**

In [20]:
wb.record("CP3.14a", FITS_14, mistakes={"n'oublie pas le réentraînement final du meilleur candidat (refit=True)": FITS_14 - 1,
                     "relis le décompte : une grille essaie le PRODUIT des nombres de valeurs, et chaque candidat est entraîné une fois PAR FOLD": CANDIDATES_14 + 1,
                     "c'est le nombre de candidats : chacun est entraîné une fois par fold, plus le réentraînement final": CANDIDATES_14})
wb.record("CP3.14b", round(FITS_14 * 2 / 60), fractional="arrondis à l'entier le plus proche", mistakes={"c'est une durée en secondes : convertis-la en minutes": FITS_14 * 2})
wb.record("CP3.14c", 5 * FITS_14, mistakes={"chaque tour extérieur refait toute la recherche, réentraînement final compris": 5 * (FITS_14 - 1),
                     "l'énoncé ne compte pas la recherche finale qui produirait le modèle livré": 6 * FITS_14})
wb.record("CP3.14d", 12 * 5 + 1, mistakes={"n'oublie pas le réentraînement final": 12 * 5,
                     "chaque candidat est entraîné une fois PAR FOLD": 12 + 1})

WB_ANSWER {"hash": "4e934ddf29ce3daf82014bf705afa378ac0f6c490a70451b49a99bc8c8b2beeb", "id": "CP3.14a", "kind": "int", "magnitude": 2, "mistakes": {"33518291127eab043a806223986c4e743b4b3bc7568314fd2f61cb81b650eb26": "n'oublie pas le réentraînement final du meilleur candidat (refit=True)", "c97b458333af6f5e34f322590be108e46ad4a955aa3100b9d4f0c53cd83c3cbb": "c'est le nombre de candidats : chacun est entraîné une fois par fold, plus le réentraînement final", "d1fe05818d82c721432d997a55d383ccad95e1f17c5f42b20105fffd0c61e316": "relis le décompte : une grille essaie le PRODUIT des nombres de valeurs, et chaque candidat est entraîné une fois PAR FOLD"}, "sign": 1}
📝 Ex CP3.14a : réponse enregistrée (int).
WB_ANSWER {"fractional": "arrondis à l'entier le plus proche", "hash": "0eb91fd9cd0a4e4e63df1f960889a216410dbd8f83bdf513f54a26647958b0ea", "id": "CP3.14b", "kind": "int", "magnitude": 1, "mistakes": {"db371884975d59fb6ee1e137b321fa83c05f86b6edd9df296bc9a509f3444dd8": "c'est une durée en seco